# 10 - Streaming and Real-Time Processing

**Objective:** Implement IQ streaming, on-FPGA averaging, and real-time decimation. This notebook covers:
- Streaming mode for continuous IQ data capture
- On-FPGA averaging for noise reduction
- Real-time decimation to reduce data bandwidth
- Comparison of streaming vs. buffered readout modes

## 1. Setup

In [ ]:
# Standard imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import logging
from scipy import signal

from qick import *
from qick.asm_v2 import AveragerProgramV2

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

# Connect to the board
BITSTREAM_PATH = '/path/to/your/firmware.bit'  # <--- CHANGE THIS
soc = QickSoc(BITSTREAM_PATH)
soccfg = soc

# Define hardware channels
GEN_CH = 0
RO_CH = 0

print("Ready")

## 2. Streaming Mode Overview

By default, the QICK captures readout data in **buffered mode**, where data is stored in on-board memory and transferred after the experiment. 

**Streaming mode** continuously streams IQ data from the ADC to the host computer in real-time. This is useful for:
- Monitoring signals over long periods
- Real-time feedback and decision making
- Capturing transient events
- High-throughput experiments

The trade-off: streaming generates large amounts of data and requires high-bandwidth communication.

## 3. Basic Streaming Example

Let's configure a simple streaming acquisition.

In [ ]:
class StreamingProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        
        # Enable streaming mode
        self.stream_enable(True)
        
        # Configure streaming channel with decimation
        self.stream_channel(
            ch=cfg['ro_ch'],
            decimation=cfg['decimation'],  # Reduce data rate by factor
            ddr4=cfg.get('use_ddr4', False)  # Use DDR4 for large streams
        )
        
        # Define a pulse
        self.add_pulse(
            ch=cfg['gen_ch'], name="test_pulse",
            style="const",
            freq=cfg['freq'], length=cfg['pulse_len'],
            phase=0, gain=cfg['gain']
        )

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

config_stream = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100.0,
    'pulse_len': 1.0,  # 1 µs pulse
    'gain': 0.5,
    'decimation': 1,    # No decimation (full rate)
    'use_ddr4': False
}

prog = StreamingProgram(soccfg, reps=1, final_delay=0.5, cfg=config_stream)

# Run streaming acquisition
stream_data = prog.acquire_stream(soc, length=2000)  # Capture 2000 samples

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(stream_data['i'][:500], label='I')
plt.plot(stream_data['q'][:500], label='Q')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude (ADC counts)')
plt.title('Streaming Mode - Raw IQ Data')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(stream_data['i'][:500] + 1j * stream_data['q'][:500], '.')
plt.xlabel('I')
plt.ylabel('Q')
plt.title('Streaming Mode - IQ Constellation')
plt.axis('equal')
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Total samples captured: {len(stream_data['i'])}")
print(f"Data rate: {len(stream_data['i']) / config_stream['pulse_len'] * 1e-6:.1f} MSamples/s")

## 4. Decimation in Streaming Mode

Decimation reduces the data rate by averaging or downsampling. This is crucial for long experiments where you don't need the full sample rate.

In [ ]:
# Compare different decimation factors
decimations = [1, 4, 16, 64]
results = {}

for dec in decimations:
    config = config_stream.copy()
    config['decimation'] = dec
    config['pulse_len'] = 2.0  # Longer pulse to see decimation effect
    
    prog = StreamingProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
    stream_data = prog.acquire_stream(soc, length=500)
    results[dec] = stream_data

plt.figure(figsize=(12, 8))
for i, dec in enumerate(decimations):
    plt.subplot(2, 2, i+1)
    data = results[dec]
    samples_to_plot = min(200, len(data['i']))
    plt.plot(data['i'][:samples_to_plot], label=f'I (dec={dec})')
    plt.plot(data['q'][:samples_to_plot], label=f'Q (dec={dec})')
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.title(f'Decimation Factor: {dec}\n({len(data["i"])} total samples)')
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()

print("Decimation Summary:")
for dec in decimations:
    print(f"  dec={dec}: {len(results[dec]['i'])} samples, "
          f"effective rate = {100e6 / dec:.1f} Hz")
print("\nNote: Higher decimation reduces data volume but also temporal resolution.")

## 5. Real-Time Averaging

The QICK FPGA can perform on-the-fly averaging of multiple shots. This reduces noise without transferring all raw data to the host.

In [ ]:
class AveragingProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Enable on-FPGA averaging
        # This accumulates data across multiple shots before transfer
        self.enable_avg(True)
        
        # Configure the number of averages to perform
        self.set_avg_mode(
            mode='soft',           # 'soft' for Python-controlled, 'hard' for FPGA
            rounds=cfg['avg_rounds']
        )

        self.add_pulse(ch=cfg['gen_ch'], name="test_pulse",
                       style="const",
                       freq=cfg['freq'], length=cfg['pulse_len'],
                       phase=0, gain=cfg['gain'])

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq'], gen_ch=cfg['gen_ch'])

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

config_avg = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100.0,
    'pulse_len': 0.3,
    'gain': 0.5,
    'trig_time': 0.05,
    'ro_len': 0.5,
    'avg_rounds': 100
}

prog = AveragingProgram(soccfg, reps=1, final_delay=0.5, cfg=config_avg)

# Acquire with averaging
iq_avg = prog.acquire_decimated(soc, rounds=config_avg['avg_rounds'])
time_axis = prog.get_time_axis(ro_index=0)

plt.figure(figsize=(10, 5))
plt.plot(time_axis, np.abs(iq_avg[0].dot([1, 1j])), label='Averaged magnitude')
plt.xlabel('Time (µs)')
plt.ylabel('|IQ| (a.u.)')
plt.title(f'On-FPGA Averaging: {config_avg["avg_rounds"]} shots')
plt.grid(True)
plt.show()

print(f"Averaged data shape: {iq_avg.shape}")
print("Note: The FPGA accumulates data and returns the average, significantly reducing noise.")

## 6. Streaming vs. Buffered Readout Comparison

Understanding the difference between streaming and buffered modes helps you choose the right approach for your experiment.

In [ ]:
def compare_modes():
    """Compare streaming mode vs. buffered mode for the same pulse."""
    
    # Streaming mode
    class StreamCompare(AveragerProgramV2):
        def _initialize(self, cfg):
            self.declare_gen(ch=cfg['gen_ch'], nqz=1)
            self.stream_enable(True)
            self.stream_channel(ch=cfg['ro_ch'], decimation=1)
            
            self.add_pulse(ch=cfg['gen_ch'], name="pulse",
                           style="const", freq=cfg['freq'],
                           length=cfg['pulse_len'], phase=0, gain=cfg['gain'])

        def _body(self, cfg):
            self.pulse(ch=cfg['gen_ch'], name="pulse", t=0)
    
    # Buffered mode
    class BufferCompare(AveragerProgramV2):
        def _initialize(self, cfg):
            self.declare_gen(ch=cfg['gen_ch'], nqz=1)
            self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
            
            self.add_pulse(ch=cfg['gen_ch'], name="pulse",
                           style="const", freq=cfg['freq'],
                           length=cfg['pulse_len'], phase=0, gain=cfg['gain'])
            
            self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                                   freq=cfg['freq'], gen_ch=cfg['gen_ch'])

        def _body(self, cfg):
            self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
            self.pulse(ch=cfg['gen_ch'], name="pulse", t=0)
    
    common_config = {
        'gen_ch': GEN_CH,
        'ro_ch': RO_CH,
        'freq': 100.0,
        'pulse_len': 0.5,
        'gain': 0.5,
        'ro_len': 0.6,
        'trig_time': 0.05
    }
    
    # Run streaming
    prog_stream = StreamCompare(soccfg, reps=1, final_delay=0.5, cfg=common_config)
    stream = prog_stream.acquire_stream(soc, length=1000)
    
    # Run buffered
    prog_buffer = BufferCompare(soccfg, reps=1, final_delay=0.5, cfg=common_config)
    iq_buffer = prog_buffer.acquire_decimated(soc, rounds=1)
    time_axis = prog_buffer.get_time_axis(ro_index=0)
    
    # Plot comparison
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(stream['i'][:500], label='I (stream)')
    plt.plot(stream['q'][:500], label='Q (stream)')
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.title('Streaming Mode')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(time_axis, iq_buffer[0][:,0], label='I (buffer)')
    plt.plot(time_axis, iq_buffer[0][:,1], label='Q (buffer)')
    plt.xlabel('Time (µs)')
    plt.ylabel('Amplitude')
    plt.title('Buffered Mode')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    return stream, iq_buffer

# Uncomment to run comparison:
# stream_data, buffer_data = compare_modes()

## 7. Practical Application: Real-Time Power Monitoring

Streaming data can be processed in real-time for feedback applications. Here's an example of monitoring signal power and making decisions.

In [ ]:
def real_time_power_monitoring(threshold=1000, duration_ms=10):
    """
    Monitor signal power in real-time and detect when it exceeds a threshold.
    
    Parameters:
    - threshold: Power threshold for detection (ADC counts^2)
    - duration_ms: Duration to monitor in milliseconds
    """
    
    class MonitorProgram(AveragerProgramV2):
        def _initialize(self, cfg):
            self.declare_gen(ch=cfg['gen_ch'], nqz=1)
            self.stream_enable(True)
            self.stream_channel(ch=cfg['ro_ch'], decimation=1)
            
            # Generate a continuous pulse for monitoring
            self.add_pulse(ch=cfg['gen_ch'], name="cw_pulse",
                           style="const", freq=cfg['freq'],
                           length=cfg['duration'], phase=0, gain=cfg['gain'])

        def _body(self, cfg):
            # Play a long CW pulse
            self.pulse(ch=cfg['gen_ch'], name="cw_pulse", t=0)
    
    # Calculate number of samples needed
    sample_rate = 100e6  # 100 MHz ADC
    num_samples = int(duration_ms * 1e-3 * sample_rate)
    
    config = {
        'gen_ch': GEN_CH,
        'ro_ch': RO_CH,
        'freq': 100.0,
        'duration': duration_ms / 1000.0,  # Convert to seconds
        'gain': 0.3
    }
    
    prog = MonitorProgram(soccfg, reps=1, final_delay=0, cfg=config)
    
    # Stream data
    # Note: In practice, you would process chunks as they arrive
    stream_data = prog.acquire_stream(soc, length=min(num_samples, 10000))
    
    # Calculate power
    iq = stream_data['i'] + 1j * stream_data['q']
    power = np.abs(iq)**2
    
    # Detect threshold crossings
    threshold_crossings = np.where(power > threshold)[0]
    
    # Visualization
    plt.figure(figsize=(12, 6))
    plt.subplot(2, 1, 1)
    plt.plot(stream_data['i'][:1000], label='I')
    plt.plot(stream_data['q'][:1000], label='Q')
    plt.axhline(y=np.sqrt(threshold), color='r', linestyle='--', 
                label=f'Threshold (±{np.sqrt(threshold):.0f})')
    plt.ylabel('Amplitude (ADC)')
    plt.title('Real-Time Monitoring - Raw IQ')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    plt.plot(power[:1000], label='Instantaneous Power')
    plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({threshold})')
    if len(threshold_crossings) > 0:
        plt.scatter(threshold_crossings[threshold_crossings < 1000], 
                   power[threshold_crossings[threshold_crossings < 1000]], 
                   color='r', s=50, zorder=5, label='Threshold Crossings')
    plt.xlabel('Sample Index')
    plt.ylabel('Power (ADC²)')
    plt.title('Power Monitoring')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    print(f"Monitoring duration: {duration_ms} ms")
    print(f"Samples captured: {len(power)}")
    print(f"Threshold crossings: {len(threshold_crossings)}")
    
    return power, threshold_crossings

# Uncomment to run:
# power, crossings = real_time_power_monitoring(threshold=500, duration_ms=5)

## 8. Summary

You have learned:
- How to use streaming mode for continuous IQ data capture
- How decimation reduces data rate while preserving signal information
- How on-FPGA averaging reduces noise without host involvement
- The trade-offs between streaming and buffered modes
- How to implement real-time power monitoring

**Key takeaways:**
- Streaming is ideal for long acquisitions and real-time feedback
- Decimation allows you to balance data rate and temporal resolution
- On-FPGA averaging is efficient for repetitive measurements
- Choose buffered mode for short, high-resolution acquisitions

**Next steps:** Proceed to [`11_DSP_Blocks_And_Correlators.ipynb`](./11_DSP_Blocks_And_Correlators.ipynb) to learn about advanced signal processing with DSP blocks and correlators.